In [1]:
import pandas as pd
import numpy as np
import re
from functools import reduce

In [2]:
columns = ['id','source','lang','comment_text','toxic']

### Train

In [3]:
data1 = pd.read_csv('../../data/raw/jigsaw-toxic-comment-train.csv')
data2 = pd.read_csv('../../data/raw/jigsaw-unintended-bias-train.csv')
data3 = pd.read_csv('../../data/raw/extra_english.csv')

In [4]:
labels = ['toxic', 'severe_toxicity']

In [5]:
data1['filter'] = 1.
data2['filter'] = (data2[labels].sum(axis=1) > 0).astype(int)
data3['filter'] = 1.

In [6]:
data2['toxic'] = (data2['toxic'] >= 0.5).astype(int)

In [7]:
data1['source'] = '2020-train'
data2['source'] = '2019-train'
data1['lang'] = 'en'
data2['lang'] = 'en'

In [8]:
data1 = data1[columns + ['filter']]
data2 = data2[columns + ['filter']]
data3 = data3[columns + ['filter']]

In [9]:
data = data1.append(data2).append(data3)

In [10]:
data['source'].value_counts()

2019-train    1902194
2020-train     223549
prev-test        9397
Name: source, dtype: int64

In [11]:
len(data), data['toxic'].sum(), data['toxic'].mean()

(2135140, 182892, 0.08565808331069626)

In [12]:
data = data[data['filter'] > 0]

In [13]:
len(data), data['toxic'].sum(), data['toxic'].mean()

(802105, 182892, 0.22801503543800375)

In [14]:
data = data.drop('filter', axis=1)

In [15]:
data.head()

,id,source,lang,comment_text,toxic
0,0000997932d777bf,2020-train,en,Explanation\nWhy the edits made under my usern...,0
1,000103f0d9cfb60f,2020-train,en,D'aww! He matches this background colour I'm s...,0
2,000113f07ec002fd,2020-train,en,"Hey man, I'm really not trying to edit war. It...",0
3,0001b41b1c6bb37e,2020-train,en,"""\nMore\nI can't make any real suggestions on ...",0
4,0001d958c54c6e35,2020-train,en,"You, sir, are my hero. Any chance you remember...",0


In [16]:
data['source'].value_counts()

2019-train    569159
2020-train    223549
prev-test       9397
Name: source, dtype: int64

In [17]:
data.to_csv('../../data/process/english/train_english.csv', index=False)

### Valid

In [18]:
data1 = pd.read_csv('../../data/raw/jigsaw_multilingual_valid_google.csv')
data2 = pd.read_csv('../../data/raw/jigsaw_multilingual_valid_yandex.csv')

In [19]:
data1 = data1.drop('comment_text', axis=1)
data2 = data2.drop('comment_text', axis=1)

In [20]:
data1 = data1.rename(columns={'comment_text_en' : 'comment_text'})
data2 = data2.rename(columns={'translated' : 'comment_text'})

In [21]:
data1['source'] = '2020-valid-google'
data2['source'] = '2020-valid-yandex'
data1['lang'] = 'en'
data2['lang'] = 'en'

In [22]:
data1 = data1[columns]
data2 = data2[columns]

In [23]:
data = data1.append(data2)
data['original'] = 0

In [24]:
data['source'].value_counts()

2020-valid-google    8000
2020-valid-yandex    8000
Name: source, dtype: int64

In [25]:
len(data), data['toxic'].sum(), data['toxic'].mean()

(16000, 2460, 0.15375)

In [26]:
data.head()

,id,source,lang,comment_text,toxic,original
0,0,2020-valid-google,en,This user does not even rank heretic. Therefor...,0,0
1,1,2020-valid-google,en,The text of this item seems to be plagiarized ...,0,0
2,2,2020-valid-google,en,OK. I'm just stating my past. All past time wa...,1,0
3,3,2020-valid-google,en,I var.ön My hesitation about continuing issues...,0,0
4,4,2020-valid-google,en,Belgium's towns and villages while the city ne...,0,0


In [27]:
data.to_csv('../../data/process/english/valid_english.csv', index=False)

### Test

In [28]:
data1 = pd.read_csv('../../data/raw/jigsaw_multilingual_test_google.csv')
data2 = pd.read_csv('../../data/raw/jigsaw_multilingual_test_yandex.csv')

In [29]:
data1 = data1.drop('content', axis=1)
data2 = data2.drop('content', axis=1)

In [30]:
data1 = data1.rename(columns={'content_en' : 'comment_text'})
data2 = data2.rename(columns={'translated' : 'comment_text'})

In [31]:
data1['source'] = '2020-test-google'
data2['source'] = '2020-test-yandex'
data1['lang'] = 'en'
data2['lang'] = 'en'

In [32]:
data1 = data1[columns[:4]]
data2 = data2[columns[:4]]

In [33]:
data = data1.append(data2)
data['original'] = 0

In [34]:
data['source'].value_counts()

2020-test-google    63812
2020-test-yandex    63812
Name: source, dtype: int64

In [35]:
len(data)

127624

In [36]:
data.head()

,id,source,lang,comment_text,original
0,0,2020-test-google,en,Doctor Who has a wiki-wiki title in the 12th d...,0
1,1,2020-test-google,en,"Quite possibly, but I do not see the need to ...",0
2,2,2020-test-google,en,"So you're one of those conservative, preferrin...",0
3,3,2020-test-google,en,"Unfortunately, however, he had not done someth...",0
4,4,2020-test-google,en,Picture: Seldabagcan.jpg the official source o...,0


In [37]:
data.to_csv('../../data/process/english/test_english.csv', index=False)